In [0]:
from pyspark.sql.functions import col, count

SILVER_TABLE = "workspace.default.silver_hvfhv_trips"

df_silver = spark.table(SILVER_TABLE)

display(
    df_silver
    .groupBy("hvfhs_license_num")
    .agg(count("*").alias("trip_count"))
    .orderBy(col("trip_count").desc())
)

In [0]:
from pyspark.sql.functions import col, when

SILVER_TABLE = "workspace.default.silver_hvfhv_trips"
PROVIDER_TABLE = "workspace.default.dim_provider"

df_silver = spark.table(SILVER_TABLE)

df_provider = (
    df_silver
    .select("hvfhs_license_num")
    .distinct()
    .withColumn(
        "provider_name",
        when(col("hvfhs_license_num") == "HV0002", "Juno")
        .when(col("hvfhs_license_num") == "HV0003", "Uber")
        .when(col("hvfhs_license_num") == "HV0004", "Via")
        .when(col("hvfhs_license_num") == "HV0005", "Lyft")
        .otherwise("Unknown")
    )
    .withColumn(
        "provider_key",
        col("hvfhs_license_num")
    )
    .select(
        "provider_key",
        col("hvfhs_license_num").alias("provider_code"),
        "provider_name"
    )
)

In [0]:
print("Provider rows:", df_provider.count())
print("Provider columns:", len(df_provider.columns))

display(
    df_provider.orderBy("provider_code")
)

In [0]:
(
    df_provider
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(PROVIDER_TABLE)
)

In [0]:
display(
    spark.table(PROVIDER_TABLE)
    .orderBy("provider_code")
)